In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [4]:
block_size = 3 

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size 
        for ch in w + '.': 
            ix = stoi[ch]  
            X.append(context) 
            Y.append(ix)
            context = context[1:] + [ix]  
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])    # 训练集
Xdev, Ydev = build_dataset(words[n1:n2])# 验证集
Xte, Yte = build_dataset(words[n2:])    # 测试集

In [5]:
#比较手动梯度计算与 pytorch自动计算梯度
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item() #精确比较
    app = torch.allclose(dt, t.grad)    #估计比较
    maxdiff = (dt - t.grad).abs().max().item() #手动梯度计算与自动计算的差异值
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [11]:
def cmp(s, dt, t):
    if t.grad is None:
        print(f'{s:15s} | ERROR: t.grad is None')
        return
    
    # 确保形状匹配
    if dt.shape != t.grad.shape:
        print(f'{s:15s} | ERROR: shape mismatch dt{dt.shape} vs grad{t.grad.shape}')
        return
    
    # 确保都是张量
    dt = dt.to(t.grad.device).to(t.grad.dtype)
    
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [6]:
n_embd = 10 
n_hidden = 200 

g = torch.Generator().manual_seed(2147483647) 
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)* (5/3)/(n_embd * block_size)**0.5
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 
W2 = torch.randn((n_hidden, vocab_size),          generator=g)* 0.1 
b2 = torch.randn(vocab_size,                      generator=g)* 0.1 

bngain = torch.ones((1,n_hidden))* 0.1 + 1.0 
bnbias = torch.zeros((1,n_hidden))* 0.1 

bnmean_running = torch.zeros((1,n_hidden))
bnstd_running = torch.ones((1,n_hidden))

parameters = [C, W1, W2, b1 ,b2 , bngain,bnbias] 
print(sum(p.nelement() for p in parameters)) 
for p in parameters:
  p.requires_grad = True 

12297


In [7]:
batch_size = 32  # 批次大小
n = batch_size  # 用一个更短的变量名，方便使用
# 构建一个小批量（minibatch）
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)  # 随机生成 batch_size 个索引，范围在 [0, 训练集样本数)
Xb, Yb = Xtr[ix], Ytr[ix]  # 根据索引取出对应的输入 X 和标签 Y，组成一个批次

In [25]:
# 前向传播，将计算拆解为多个小步骤，以便逐个进行反向传播
emb = C[Xb]  # 将字符索引嵌入为向量
embcat = emb.view(emb.shape[0], -1)  # 将向量拼接（展平）
#-------------------------------------------------------------
# 第一层线性变换
hprebn = embcat @ W1 + b1  # 隐藏层预激活值 -torch.Size([32, 200])
#-------------------------------------------------------------
# 批归一化层（BatchNorm）
bnmeani = 1/n * hprebn.sum(0, keepdim=True)  # 计算均值 - torch.Size([1, 200])
#bnmeani = hprebn.sum(0, keepdim=True) / n 
bndiff = hprebn - bnmeani  # 减去均值-torch.Size([32, 200])
bndiff2 = bndiff**2  # 差值的平方
bnvar = 1/(n-1) * (bndiff2).sum(0, keepdim=True)  # 计算批次方差
bnvar_inv = (bnvar + 1e-5)**-0.5  # 方差的逆平方根（加 1e-5 防止除零）
bnraw = bndiff * bnvar_inv  # 标准化后的原始值
hpreact = bngain * bnraw + bnbias  # 缩放和平移（可学习的参数）
#-------------------------------------------------------------
# 非线性激活
h = torch.tanh(hpreact)  # 隐藏层输出
#-------------------------------------------------------------
# 第二层线性变换
logits = h @ W2 + b2  # 输出层（未归一化的对数概率）
#-------------------------------------------------------------
# SoftMax
logit_maxes = logits.max(1, keepdim=True).values  # 每行取最大值，用于数值稳定性
norm_logits = logits - logit_maxes  # 减去最大值（防止指数爆炸）

counts = norm_logits.exp()  # 计算未归一化的概率（指数）
counts_sum = counts.sum(1, keepdims=True)  # 每行求和
counts_sum_inv = counts_sum**-1  # 求和的倒数（如果用 1.0 / counts_sum 则反向传播会出问题）
probs = counts * counts_sum_inv  # 归一化得到概率分布
#-------------------------------------------------------------
# 负对数似然
logprobs = probs.log()  # 取对数
loss = -logprobs[range(n), Yb].mean()  # 负对数似然损失（取正确标签位置的对数概率，求平均）
#-------------------------------------------------------------
# PyTorch 反向传播
for p in parameters:
    p.grad = None  # 清空梯度
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,  
          norm_logits, logits, logit_maxes, logits, h, hpreact, bnraw,
          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
          embcat, emb]:
    t.retain_grad()  # 保留中间张量的梯度（默认只保留叶节点的梯度）
loss.backward()  # 执行反向传播
loss

tensor(3.8279, grad_fn=<NegBackward0>)

In [15]:
dlogprobs = torch.zeros_like(logprobs)          # 先创建全零张量，shape: (32, 27)
dlogprobs[range(n), Yb] = -1.0 / n               # 只在正确标签位置填入 -1/n
cmp('logprobs', dlogprobs, logprobs)

#-------------------------------------------------------------
dprobs = dlogprobs * (1.0 / probs)
cmp('probs', dprobs, probs)

#-------------------------------------------------------------
#由于counts_sum_inv与counts维度不同
#counts_sum_inv[32, 1],counts[32, 27]
#python将单一运算使用两个连续操作完成（1）广播复制（2）乘法运算
#反向传播会对于整个过程反向计算先反向推到乘法运算后，反向传播广播的复制操作。
#注意：如果一个节点被使用多次，那么在反向传播中它所有使用的路径梯度会被求和
#这就是广播操作反向传播需要求和的本质
dcounts_sum_inv = (dprobs * counts).sum(1, keepdims=True) 
dcounts_sum = dcounts_sum_inv * -counts_sum**-2
#多元链式法则：当一个变量通过多条路径影响输出时，梯度要累加。
dcounts= dprobs * counts_sum_inv + dcounts_sum
cmp('counts', dcounts, counts)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

#-------------------------------------------------------------
#dcounts_sum = dcounts_sum_inv * -counts_sum**-2
cmp('counts_sum', dcounts_sum, counts_sum)

#-------------------------------------------------------------
dnorm_logits = dcounts * norm_logits.exp()
cmp('norm_logits', dnorm_logits, norm_logits)

#-------------------------------------------------------------
dlogit_maxes = (-dnorm_logits).sum(1, keepdims=True) 
dlogits = dnorm_logits
#-------------------------------------------------------------
max_indices = logits.max(1, keepdim=True).indices 
mask = torch.zeros_like(logits)                     
mask.scatter_(1, max_indices, 1)                     
dlogits += dlogit_maxes * mask  
cmp('logits', dlogits, logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)


#-------------------------------------------------------------\
db2 = dlogits.sum(0) 
dW2 = h.T @ dlogits 
dH = dlogits @ W2.T 

cmp('h', dH, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

#-------------------------------------------------------------
dhpreact = dH * (1-h**2)
cmp('hpreact', dhpreact, hpreact)

#-------------------------------------------------------------
dbnbias = dhpreact.sum(0, keepdim=True)
cmp('bnbias', dbnbias, bnbias)

#-------------------------------------------------------------
dbngain = (bnraw*dhpreact).sum(0, keepdim=True)
cmp('bngain', dbngain, bngain)

#-------------------------------------------------------------
dbnraw = dhpreact*bngain
cmp('bnraw', dbnraw, bnraw)

#-------------------------------------------------------------
dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)

#-------------------------------------------------------------
dbndiff = dbnraw * bnvar_inv
dbnvar = dbnvar_inv * -0.5*(bnvar + 1e-5)**-1.5
dbndiff2 = (dbnvar * (1.0 / (n-1))).expand_as(bndiff2) 
dbndiff+=dbndiff2 * 2 * bndiff 
cmp('bndiff', dbndiff, bndiff)

#-------------------------------------------------------------
# dbnvar = dbnvar_inv * -0.5*(bnvar + 1e-5)**-1.5
cmp('bnvar', dbnvar, bnvar)

# dbndiff2 = (dbnvar * (1.0 / (n-1))).expand_as(bndiff2) 
cmp('bndiff2', dbndiff2, bndiff2)

#-------------------------------------------------------------
dbnmeani = -dbndiff.sum(0, keepdim=True)
dhprebn = dbndiff
dhprebn += 1/n *dbnmeani.expand_as(dhprebn)

cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)

#-------------------------------------------------------------
db1 = dhprebn.sum(0)
dW1 = embcat.T @ dhprebn
dembcat =dhprebn @ W1.T  
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)

#-------------------------------------------------------------
demb = dembcat.view(n, block_size, n_embd)
cmp('emb', demb, emb)

#-------------------------------------------------------------
#嵌入操作对于前向传播过程来说就是将输入样本（contxt = 3）其所对应的属性（10）打包进张量emb（32，3，10）
#那么反向传播就是在我们得到emb中每一个值（属性）的梯度的情况下，找到每一个值（属性）对应的字符对应的在C矩阵中的属性传递累加回去

dC = torch.zeros_like(C)   # (27, 10)

for i in range(n):                # 遍历小批次样本
    for j in range(block_size):   # 遍历每一个样本的上下文（字符）
        #遍历Xb中的所有元素
        ix = Xb[i, j]            # 得到每一个样本中每一个字符
        dC[ix] += demb[i, j]     # 找到矩阵C对应该字符这一行的属性传递梯度

cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [24]:
bndiff.shape,bnvar_inv.shape,bnraw.shape

(torch.Size([32, 200]), torch.Size([1, 200]), torch.Size([32, 200]))